In [2]:
import csv
import os
import random
import time
from PIL import Image
import PIL
import cv2
import imutils
import numpy as np
from tqdm import tqdm



In [3]:
def getPlateName(n1, n2, l, n3, n4, n5, c1, c2,base = False):
    if base:
        return f'{n1}{n2}{l}{n3}{n4}{n5}_{c1}{c2}'
    else:
        return f'{n1}{n2}_{n3}{n4}{n5}'

In [4]:
def getNewPlate(template_name , available_letters = ["sin", "sad", "beh", "jim", "dal", "ta", "ghaf", "lam","mim","noon","vav","he", "ye"], numbers = [0,1,2,3,4,5,6,7,8,9]):
    # ب ج د س
    # ص ط ق ل
    # م ن و ه‍ ی
    """
    تولید ساختار کامل پلاک (۸ کاراکتر):
    [n1, n2, letter, n3, n4, n5, city1, city2]
    """
    valid_letters = available_letters.copy()
    selected_letter = random.choice(valid_letters) if valid_letters else random.choice(available_letters)

    return [
        random.choice(range(1, 10)),  # n1
        random.choice(range(1, 10)),  # n2
        selected_letter,         # letter
        random.choice(range(1, 10)),  # n3
        random.choice(range(1, 10)),  # n4
        random.choice(range(1, 10)),  # n5
        random.choice(range(1, 10)),  # city_code_1
        random.choice(numbers)   # city_code_2
    ]

In [5]:
def getGlyphAddress(font, glyphName):
    t = "C:\\Users\\MEHDI\\Desktop\\Projects\\ALPR-Persian\\ocr\\Dataset\\"
    return f'{t}Glyphs/{font}/{glyphName}_trim.png'


In [52]:
cursor = {
    "dolati": {"x":65,"y":32,"s_2r":6, "s_3r":6, "s_l":32},
    "gozar": {"x":65,"y":32,"s_2r":5, "s_3r":5, "s_l":30},
    "nezami": {"x":73,"y":32,"s_2r":5, "s_3r":5, "s_l":27},
    "malolin va janbazan": {"x":70,"y":32,"s_2r":6, "s_3r":7, "s_l":24},
    "shakhsi": {"x":73,"y":32,"s_2r":5, "s_3r":6, "s_l":23},
    "siasi": {"x":70,"y":32,"s_2r":5, "s_3r":8, "s_l":22},
    "tranzit": {"x":50,"y":5,"s_2r":5, "s_3r":7, "s_l":19},
    "tashrifat": {"x":332,"y":32,"s_2r":4, "s_3r":4, "s_l":-29},
    "ommomi": {"x":65,"y":32,"s_2r":6, "s_3r":7, "s_l":30},
}

In [66]:
# Characters of Letters and Numbers in Plates
numbers = [str(i) for i in range(0, 10)]
# Fonts and Templates
font = 'roya_bold'
t = "C:\\Users\\MEHDI\\Desktop\\Projects\\ALPR-Persian\\ocr\\Dataset\\Templates\\"
for templates in os.listdir(t):
    base = templates in ['shakhsi','malolin va janbazan','tranzit']

    for template in os.listdir(t + templates):
                # دریافت ساختار کامل ۸ کاراکتری پلاک
        plate = getNewPlate(template)
        plateName = getPlateName(*plate,base=base)

        # خواندن تمام گلیف‌ها (اطمینان از وجود ۸ تصویر گلیف)
        glyphImages = [Image.open(getGlyphAddress(font, g)).convert("RGBA") for g in plate]

        print(templates , template)
        newPlate = Image.new('RGBA', (600, 132), (0, 0, 0, 0))
        background = Image.open(f'{t}{templates}\\{template}').convert("RGBA")
        newPlate.paste(background, (0, 0))
        x_cursor = cursor[templates]["x"]
        y_cursor = cursor[templates]["y"]
        s_2r = cursor[templates]["s_2r"]
        s_3r = cursor[templates]["s_3r"]    
        s_l = cursor[templates]["s_l"]
        newPlate.paste(glyphImages[0], (x_cursor, y_cursor), mask=glyphImages[0])
        x_cursor += 50 + s_2r 
        # print(x_cursor)
        newPlate.paste(glyphImages[1], (x_cursor, y_cursor), mask=glyphImages[1])
        x_cursor += 45 + s_l  # فاصله تا حرف
        # print(x_cursor)
        # ۲. بخش حرف وسط
        if base and templates != 'malolin va janbazan':
            newPlate.paste(glyphImages[2], (x_cursor, y_cursor+3), mask=glyphImages[2])
        x_cursor += 65 + s_l  # فاصله تا ۳ رقم راست
        # print("size",glyphImages[2].size[0])
        
        for idx in [3, 4, 5]:
            newPlate.paste(glyphImages[idx], (x_cursor, y_cursor), mask=glyphImages[idx])
            x_cursor += 50 + s_3r
            # print(x_cursor)
        # ۴. بخش ۲ رقم کد شهر (مختصات دقیق و ثابت کادر مربع سمت راست)
        if base:
            x_city_start = 480
            newPlate.paste(glyphImages[6], (x_city_start, y_cursor), mask=glyphImages[6])
            # print(x_cursor)
            x_city_second = x_city_start + 50 + s_2r
            newPlate.paste(glyphImages[7], (x_city_second, y_cursor), mask=glyphImages[7])
            # print(x_city_second)
        Image.Image.show(newPlate)

dolati template-dolati.png
gozar template-gozar.png
malolin va janbazan malolin.png
nezami template-artesh.png
nezami template-defa.png
nezami template-Niromosalah.png
nezami template-police.png
nezami template-sepah.png
ommomi keshavarzi.png
ommomi template-ommomi.png
ommomi template-taxi.png
shakhsi template-base.png
siasi template-diplomat.png
siasi template-service.png
tashrifat template-tashrifat.png
tranzit template-treanzit.png
